# 01 HEST-1k Human Visium Size-Factor 结果总结

这个 notebook 用来记录当前已经真实跑完的 HEST-1k human Visium size-factor predictor 结果。

注意：这里不会重新下载数据、重新提 HIPT 特征、重新训练模型。下面的代码只读取本地已经生成的 `results/`、`data/HEST-1k/manifests/` 和 `data/HEST-1k/splits/` 文件，方便复核数字、解释指标含义，并作为后续实验的结果总结入口。

当前阶段只完成了 **Step 1 的初版 leave-slide-out SF predictor**。它不是最终论文结论，还需要 leave-cohort-out、leave-organ-out、空间预测图和 downstream H&E-to-ST count-scale 验证。

## 当前数据状态

已完成：

| 项目 | 当前状态 |
| --- | --- |
| human Visium slides | 421 |
| raw metadata / st / patches / thumbnails | 已下载 |
| WSI 大图 | 尚未下载 |
| barcode-aligned processed arrays | 421 slides |
| HIPT ViT-256 spot features | 421 slides |
| feature dimension | 384 |
| training-ready manifest | 421 slides |
| leave-slide-out split | train 294 / val 41 / test 86 |

关键对齐原则：使用 `patch_barcodes ∩ h5ad.obs_names`，并按 patch barcode 顺序输出 `features.npy`、`counts.npz`、`coords.npy`、`size_factor.npy`。

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
RESULT_ROOT = ROOT / "results" / "hest1k_human_visium_sf"
MANIFEST = ROOT / "data" / "HEST-1k" / "manifests" / "human_visium_sf_manifest.csv"
SPLIT_DIR = ROOT / "data" / "HEST-1k" / "splits"

manifest = pd.read_csv(MANIFEST)
feature_report = pd.read_csv(RESULT_ROOT / "patch_feature_extraction_hipt256.csv")
processed_audit = pd.read_csv(RESULT_ROOT / "processed_slide_audit_after_hipt.csv")
baselines = pd.read_csv(RESULT_ROOT / "baselines" / "sf_baselines_hipt256_leave_slide_out.csv")

print("manifest shape:", manifest.shape)
print("split counts:")
display(manifest["split"].value_counts().rename_axis("split").reset_index(name="n_slides"))
print("feature extraction status:")
display(feature_report["status"].value_counts().rename_axis("status").reset_index(name="n_slides"))
print("processed audit status:")
display(processed_audit["status"].value_counts().rename_axis("status").reset_index(name="n_slides"))

## Size factor 是什么

本项目固定使用 mean-one slide-normalized size factor：

$$\mathrm{sf}_i = \frac{\mathrm{total\ counts}_i}{\mathrm{mean}(\mathrm{total\ counts\ of\ valid\ spots\ in\ same\ slide})}$$

训练目标是：

$$\mathrm{target}_i = \log(\mathrm{sf}_i)$$

它不是一个孤立任务。后续真正要做的是 H&E-to-ST count-scale prediction：

$$\mathrm{count}_{i,g} = \mathrm{rate}_{i,g} \times \mathrm{sf}_i$$

所以 SF predictor 的价值要看它能不能帮助恢复 count scale，尤其是高 RNA 总量区域。

## 指标含义

| metric | 含义 | 怎么看 |
| --- | --- | --- |
| `log_sf_pearson` | 预测 `log(size factor)` 与真实 `log(size factor)` 的 Pearson 相关。 | 越高越好。主要看空间相对趋势和 ranking。 |
| `sf_pearson` | 把 log SF 转回 SF 后，在 mean-one 归一化空间里算 Pearson 相关。 | 越高越好。比 `log_sf_pearson` 更受极端高 SF spot 影响。 |
| `log_sf_mae` | `log(size factor)` 的平均绝对误差。 | 越低越好。比如 0.69 大致对应 exp(0.69)≈2 倍量级误差。 |
| `log_sf_rmse` | `log(size factor)` 的均方根误差。 | 越低越好。比 MAE 更惩罚大错。 |
| `sf_std_ratio` | 预测 SF 标准差 / 真实 SF 标准差。 | 理想值接近 1。小于 1 表示预测太平滑、方差不足。 |
| `sf_top_decile_mean_ratio` | 真实 top 10% 高 SF spot 内，预测平均 SF / 真实平均 SF。 | 理想值接近 1。小于 1 表示高 SF 尾部被低估。 |
| `log_sf_top_decile_mae` | 真实 top 10% 高 SF spot 的 log SF MAE。 | 越低越好。专门看高 RNA 总量区域。 |

为什么 constant baseline 的 Pearson 是 NaN：常数预测没有方差，相关系数分母为 0，所以不能定义。

In [ ]:
def load_metrics(name):
    path = RESULT_ROOT / f"sf_metrics_main_hipt256_leave_slide_out_{name}.json"
    return json.loads(path.read_text(encoding="utf-8"))

main_rows = []
for split in ["train", "val", "test"]:
    row = {"run": "MLP main HIPT", "split": split}
    row.update(load_metrics(split))
    main_rows.append(row)

smoke = json.loads((RESULT_ROOT / "sf_metrics_smoke_hipt256_test.json").read_text(encoding="utf-8"))
smoke_row = {"run": "MLP smoke 3 epoch", "split": "test", **smoke}

baseline_rows = baselines.copy()
baseline_rows.insert(0, "split", "test")
baseline_rows = baseline_rows.rename(columns={"baseline": "run"})

result_table = pd.concat([
    baseline_rows.drop(columns=["alpha"], errors="ignore"),
    pd.DataFrame([smoke_row]),
    pd.DataFrame(main_rows),
], ignore_index=True)

metric_cols = [c for c in result_table.columns if c not in {"run", "split"}]
display(result_table[["run", "split"] + metric_cols].round(4))

## 当前数字怎么解释

已经跑完的关键 test 结果：

| run | log_sf_pearson | log_sf_mae | sf_std_ratio | sf_top_decile_mean_ratio |
| --- | ---: | ---: | ---: | ---: |
| constant SF | NaN | 0.8387 | 0.0000 | 0.3116 |
| HIPT ridge | 0.4672 | 0.7549 | 0.2949 | 0.3900 |
| MLP main HIPT | 0.4913 | 0.6987 | 0.3919 | 0.5603 |

读法：

1. HIPT ridge 已经明显优于 constant baseline，说明 H&E 图像特征确实包含 SF 信号。
2. MLP main HIPT 的 test `log_sf_pearson=0.4913`，比 ridge 的 `0.4672` 略好。
3. MLP main HIPT 的 test `log_sf_mae=0.6987`，低于 ridge 的 `0.7549`，说明平均误差下降了。
4. 但是 `sf_std_ratio=0.3919`，远小于 1，说明预测仍然过于平滑。
5. `sf_top_decile_mean_ratio=0.5603`，说明真实最高 10% SF 区域里，模型平均只预测到真实强度的约 56%。这是当前最大问题。

结论：现在可以说 **有真实图像信号**，但不能说已经得到可作为主结论的 SF predictor。下一步必须做 leave-cohort-out、leave-organ-out、空间叠图 QC，以及尾部校准。

In [ ]:
plot_df = result_table[result_table["split"].eq("test")].copy()
plot_df["run"] = plot_df["run"].replace({"constant_sf": "constant SF", "hipt_ridge": "HIPT ridge"})

ax = plot_df.set_index("run")[["log_sf_mae", "log_sf_top_decile_mae"]].plot(
    kind="bar", figsize=(8, 4), rot=0, title="Lower is better: log SF error on test split"
)
ax.set_ylabel("error")
ax.figure.tight_layout()

In [ ]:
ax = plot_df.set_index("run")[["log_sf_pearson", "sf_top_decile_mean_ratio", "sf_std_ratio"]].plot(
    kind="bar", figsize=(8, 4), rot=0, title="Higher is better; ratios should approach 1"
)
ax.axhline(1.0, color="black", linewidth=1, linestyle="--", alpha=0.5)
ax.set_ylabel("metric value")
ax.figure.tight_layout()

## 已完成命令记录

这些命令已经实际运行过。这里列出来是为了让 notebook 既能总结结果，也能记录结果来源。

```powershell
conda run -n histogene_bench python scripts\hest_extract_patch_features.py --model hipt256 --overwrite --batch-size 256 --out-csv results\hest1k_human_visium_sf\patch_feature_extraction_hipt256.csv
python scripts\hest_prepare_slide_arrays.py --processed-root data\HEST-1k\processed --out-csv results\hest1k_human_visium_sf\processed_slide_audit_after_hipt.csv
python scripts\hest_build_manifest.py --config configs\hest1k_human_visium_sf.yaml
python scripts\hest_make_splits.py --write-split-manifest
conda run -n histogene_bench python scripts\run_sf_baselines.py --config configs\hest1k_human_visium_sf.yaml --output-csv results\hest1k_human_visium_sf\baselines\sf_baselines_hipt256_leave_slide_out.csv
conda run -n histogene_bench python scripts\train_sf.py --config configs\hest1k_human_visium_sf.yaml --device cuda --output-dir checkpoints\hest1k_human_visium_sf\main_hipt256_leave_slide_out
conda run -n histogene_bench python scripts\evaluate_sf.py --config configs\hest1k_human_visium_sf.yaml --checkpoint checkpoints\hest1k_human_visium_sf\main_hipt256_leave_slide_out\best.pt --splits test --out-json results\hest1k_human_visium_sf\sf_metrics_main_hipt256_leave_slide_out_test.json
```

训练早停在第 31 个 epoch。当前 best checkpoint 位于：

`checkpoints/hest1k_human_visium_sf/main_hipt256_leave_slide_out/best.pt`

## 下一步

短期优先级：

1. 做 leave-cohort-out 和 leave-organ-out SF evaluation。
2. 导出 test slide 级别预测，画 H&E 组织切片上的 true SF / predicted SF / residual 空间分布图。
3. 针对高 SF 尾部低估做 calibration 或 loss/architecture 调整。
4. 开始 expression-rate predictor。
5. 做 `No SF / Predicted SF / True SF oracle` 的 count-scale H&E-to-ST 对比。

只有当第 5 步证明 predicted SF 对 count-scale ST 预测有帮助，SF predictor 才能成为主论文故事的一部分。